In [ ]:
from scipy.signal import butter, filtfilt, find_peaks
import neurokit2 as nk
import numpy as np
import pyxdf
import matplotlib.pyplot as plt
import pandas as pd
from biosppy.signals.ppg import ppg
import heartpy as hp
import glob
import re
import os

debugging = True
def Trace(message):
    """
    Function to print messages with a specific format.
    """
    if (debugging):
        print(f"[Trace] {message}")
    else:
        pass


In [13]:
def filter_bvp(signal, lowcut=0.5, highcut=8.0, fs=400):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    if high <= low:
        print(f"Warning: Highcut frequency ({highcut}Hz) is not above lowcut frequency ({lowcut}Hz) at fs={fs}Hz. Adjusting filter or skipping.")
        return signal
    try:
        b, a = butter(4, [low, high], btype='bandpass')
        filtered = filtfilt(b, a, signal)
        return filtered
    except ValueError as ve:
        print(f"ValueError during filtering: {ve}. Returning unfiltered signal.")
        return signal

def get_hrv_metrics(bvp_segment, timestamps, fs=400):
    if len(bvp_segment) < fs * 10:
        print(f"Warning: Segment too short ({len(bvp_segment)/fs:.2f}s, need at least 10s), skipping HRV.")
        return pd.Series(dtype=float)
    try:
        filtered_segment = filter_bvp(bvp_segment, fs=fs)
        wd, m = hp.process(filtered_segment, sample_rate=fs, calc_freq=False, high_precision=True, clean_rr=True)
        peaks = wd.get('peaklist', [])
        if len(peaks) < 5:
            print(f"Warning: Not enough peaks found ({len(peaks)}, need at least 5), skipping HRV.")
            return pd.Series(dtype=float)
        # Ensure peak indices are integers and within bounds before indexing timestamps
        valid_peaks = []
        for p in peaks:
            try:
                p_int = int(p) # Convert to integer
                if 0 <= p_int < len(timestamps): # Check bounds
                    valid_peaks.append(p_int)
            except (ValueError, TypeError):
                print(f"Warning: Invalid peak value {p} encountered, skipping it.")

        if len(valid_peaks) < 2:
            print(f"Warning: Not enough valid peaks ({len(valid_peaks)}) after filtering and conversion. Original peaks count from heartpy: {len(peaks)}. Skipping HRV.")
            return pd.Series(dtype=float)
        peak_times_sec = timestamps[valid_peaks]
        ibi_ms = np.diff(peak_times_sec) * 1000
        if len(ibi_ms) < 3:
            print(f"Warning: Not enough IBIs calculated ({len(ibi_ms)}), skipping HRV.")
            return pd.Series(dtype=float)
        ibi_event_times_sec = peak_times_sec[1:]
        hrv_indices = nk.hrv({'RRI': ibi_ms, 'RRI_Time': ibi_event_times_sec}, sampling_rate=1000)
        return hrv_indices.iloc[0]
    except Exception as e:
        print(f"Error processing segment: {e}")
        return pd.Series(dtype=float)

def extract_bvp_and_markers_from_xdf(filepath):
    print(f"Attempting to load XDF: {filepath}")
    try:
        streams, header = pyxdf.load_xdf(filepath)
    except Exception as e:
        print(f"Error loading XDF file {filepath}: {e}")
        return None, None, pd.DataFrame(), {'bvp': False, 'markers': False}
    bvp_s = next((s for s in streams if s["info"]["name"][0] == "OpenSignals" and any(ch['label'][0] == 'BVP0' for ch in s['info']['desc'][0]['channels'][0]['channel'])), None)
    mark_s = next((s for s in streams if s["info"]["name"][0] == "UnityMarkers"), None)
    streams_found = {'bvp': False, 'markers': False}
    bvp_signal, bvp_ts = None, None
    df_events = pd.DataFrame()
    if bvp_s:
        bvp_data_raw = bvp_s['time_series']
        bvp_ts_raw = bvp_s['time_stamps']
        bvp_channel_idx = next((i for i, ch in enumerate(bvp_s['info']['desc'][0]['channels'][0]['channel']) if ch['label'][0] == 'BVP0'), None)
        if bvp_channel_idx is not None and bvp_data_raw.ndim == 2 and bvp_data_raw.shape[1] > bvp_channel_idx:
            bvp_signal = bvp_data_raw[:, bvp_channel_idx].astype(np.float64)
            bvp_ts = bvp_ts_raw
            streams_found['bvp'] = True
            print(f"BVP stream found and BVP0 channel extracted from {filepath}.")
        else:
            print(f"Warning: BVP0 channel not found or data format unexpected in OpenSignals stream for {filepath}.")
    else:
        print(f"Warning: BVP stream (OpenSignals with BVP0) not found in {filepath}.")
    if mark_s:
        markers_raw = mark_s['time_series']
        marker_ts_raw = mark_s['time_stamps']
        if len(markers_raw) > 0:
            df_events = pd.DataFrame({'time': marker_ts_raw, 'event': [m[0] for m in markers_raw]})
            streams_found['markers'] = True
            print(f"UnityMarkers stream found in {filepath}.")
        else:
            print(f"Warning: UnityMarkers stream found but no event data in {filepath}.")
    return bvp_signal, bvp_ts, df_events, streams_found

def process_subject_data(subject_id, baseline_filepath, task_filepath, nominal_srate=400):
    results_list = []
    condition_from_task = "Unknown"  # Default if no task file or condition can't be parsed

    # --- Determine Condition from Task File (if it exists) ---
    if task_filepath:
        match_cond = re.search(r'task-([^_]+)_', os.path.basename(task_filepath))
        if match_cond:
            condition_from_task = match_cond.group(1)
        else:
            print(f"Warning: Could not parse condition from task file name: {os.path.basename(task_filepath)} for subject {subject_id}")

    # --- 1. Process Baseline File ---
    if baseline_filepath:
        print(f"Processing BASELINE for subject {subject_id} from: {os.path.basename(baseline_filepath)}")
        bvp_baseline, ts_baseline, df_events_baseline, streams_baseline = extract_bvp_and_markers_from_xdf(baseline_filepath)
        
        print(f"UnityMarkers content for baseline file ({os.path.basename(baseline_filepath)} if baseline_filepath else 'N/A'):")
        if streams_baseline['markers'] and not df_events_baseline.empty:
            print(df_events_baseline.to_string())
        else:
            print("No markers found, marker stream empty, or baseline file not processed.")

        if streams_baseline['bvp'] and bvp_baseline is not None and ts_baseline is not None:
            print(f"Calculating baseline HRV for {subject_id} (full file)...")
            hrv_baseline_metrics = get_hrv_metrics(bvp_baseline, ts_baseline, fs=nominal_srate)
            if not hrv_baseline_metrics.empty:
                baseline_res = {'ParticipantID': subject_id, 'Condition': condition_from_task, 'Phase': 'Baseline'}
                baseline_res.update(hrv_baseline_metrics)
                results_list.append(baseline_res)
                print(f"Baseline HRV calculated for {subject_id}.")
            else:
                print(f"No HRV metrics obtained for baseline for subject {subject_id}.")
        else:
            print(f"Could not process BVP for baseline for subject {subject_id} from {os.path.basename(baseline_filepath)}.")
    else:
        print(f"No baseline filepath provided for subject {subject_id}.")

    # --- 2. Process Task File ---
    if task_filepath:
        print(f"Processing TASK for subject {subject_id}, Condition: {condition_from_task} from: {os.path.basename(task_filepath)}")
        bvp_task_full, ts_task_full, df_events_task, streams_task = extract_bvp_and_markers_from_xdf(task_filepath)

        #print(f"UnityMarkers content for task file ({os.path.basename(task_filepath)} if task_filepath else 'N/A'):")
        #if streams_task['markers'] and not df_events_task.empty:
        #    print(df_events_task.to_string())
        #else:
        #    print("No markers found, marker stream empty, or task file not processed.")

        if streams_task['bvp'] and bvp_task_full is not None and ts_task_full is not None:
            # USE ENTIRE TASK FILE BVP DATA - NO SEGMENTATION BASED ON MARKERS
            print(f"Calculating task HRV for {subject_id} (full file {len(bvp_task_full)/nominal_srate:.2f}s)...")
            hrv_task_metrics = get_hrv_metrics(bvp_task_full, ts_task_full, fs=nominal_srate)
            
            if not hrv_task_metrics.empty:
                task_res = {'ParticipantID': subject_id, 'Condition': condition_from_task, 'Phase': 'Task'}
                task_res.update(hrv_task_metrics)
                results_list.append(task_res)
                print(f"Task HRV calculated for {subject_id}.")
            else:
                print(f"No HRV metrics obtained for task phase for subject {subject_id}.")
        else:
            print(f"Could not process BVP for task for subject {subject_id} from {os.path.basename(task_filepath)}.")
    else:
        print(f"No task filepath provided for subject {subject_id}.")
        
    return pd.DataFrame(results_list) if results_list else pd.DataFrame()

all_participant_dfs = []
data_root_folder = "/home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data"
nominal_srate_main = 400
subject_subfolders = [f.path for f in os.scandir(data_root_folder) if f.is_dir()]
print(f"Found subject subfolders: {subject_subfolders}")
for subject_folder in subject_subfolders:
    subject_folder_name = os.path.basename(subject_folder)
    match_id = re.search(r'sub[ -](.+)', subject_folder_name, re.IGNORECASE)
    if not match_id:
        print(f"Warning: Could not parse subject ID from folder name '{subject_folder_name}'. Skipping.")
        continue
    current_subject_id = match_id.group(1).strip()
    print(f"\nProcessing data for subject ID: {current_subject_id} in folder: {subject_folder_name}")
    baseline_files_found = glob.glob(os.path.join(subject_folder, "*[Bb]aseline*.xdf"))
    current_baseline_filepath = None
    if not baseline_files_found:
        print(f"Warning: No baseline file found for subject {current_subject_id}. This subject may be skipped or only task data processed if a task file exists.")
    else:
        current_baseline_filepath = baseline_files_found[0]
        if len(baseline_files_found) > 1:
            print(f"Warning: Multiple baseline files found for subject {current_subject_id}. Using '{os.path.basename(current_baseline_filepath)}'.")

    task_files_found = glob.glob(os.path.join(subject_folder, f"*_task-*Fi_.xdf"))
    current_task_filepath = None
    if not task_files_found:
        task_files_found = glob.glob(os.path.join(subject_folder, f"*{current_subject_id}*task-*Fi_.xdf")) or glob.glob(os.path.join(subject_folder, f"*task-*.xdf"))
    if not task_files_found:
        print(f"Warning: No task file found for subject {current_subject_id}. This subject may be skipped or only baseline data processed.")
    else:
        current_task_filepath = task_files_found[0]
        if len(task_files_found) > 1:
            print(f"Warning: Multiple task files found for subject {current_subject_id}. Using '{os.path.basename(current_task_filepath)}'.")
    if not current_baseline_filepath and not current_task_filepath:
        print(f"Skipping subject {current_subject_id} as no baseline or task files were identified.")
        continue
    subject_hrv_df = process_subject_data(current_subject_id, current_baseline_filepath, current_task_filepath, nominal_srate=nominal_srate_main)
    if not subject_hrv_df.empty:
        all_participant_dfs.append(subject_hrv_df)
    else:
        print(f"No HRV data generated for subject {current_subject_id}.")

if not all_participant_dfs:
    print("\nNo dataframes to concatenate. Final DataFrame will be empty.")
    final_df = pd.DataFrame()
else:
    final_df = pd.concat(all_participant_dfs, ignore_index=True)
    print("\n--- Combined Results DataFrame ---")
    if not final_df.empty:
        print(final_df.head())
    else:
        print("Final DataFrame is empty after processing all subjects.")

Found subject subfolders: ['/home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 1', '/home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 2', '/home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 3', '/home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 4', '/home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 5', '/home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/Sub 6', '/home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 7']

Processing data for subject ID: 1 in folder: sub 1
Processing BASELINE for subject 1 from: sub-1_ses-1_task-Baseline_.xdf
Attempting to load XDF: /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 1/sub-1_ses-1_task-Baseline_.xdf
BVP stream found and BVP0 channel extracted from /home/mitchell/Documents/Projects/P8-Project/

/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/neurokit2/signal/signal_interpolate.py:117: NeuroKitWarning: Duplicate x values detected. Averaging their corresponding y values.
  warn(


Task HRV calculated for 1.

Processing data for subject ID: 2 in folder: sub 2
Processing BASELINE for subject 2 from: sub-2_ses-1_task-Baseline_.xdf
Attempting to load XDF: /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 2/sub-2_ses-1_task-Baseline_.xdf
BVP stream found and BVP0 channel extracted from /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 2/sub-2_ses-1_task-Baseline_.xdf.
UnityMarkers stream found in /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 2/sub-2_ses-1_task-Baseline_.xdf.
UnityMarkers content for baseline file (sub-2_ses-1_task-Baseline_.xdf if baseline_filepath else 'N/A'):
         time          event
0  894.508259  BaselineStart
Calculating baseline HRV for 2 (full file)...


/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/neurokit2/signal/signal_interpolate.py:117: NeuroKitWarning: Duplicate x values detected. Averaging their corresponding y values.
  warn(


Baseline HRV calculated for 2.
Processing TASK for subject 2, Condition: HighFi from: sub-2_ses-1_task-HighFi_.xdf
Attempting to load XDF: /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 2/sub-2_ses-1_task-HighFi_.xdf
BVP stream found and BVP0 channel extracted from /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 2/sub-2_ses-1_task-HighFi_.xdf.
Calculating task HRV for 2 (full file 508.00s)...
Task HRV calculated for 2.

Processing data for subject ID: 3 in folder: sub 3
Processing BASELINE for subject 3 from: sub-3_ses-1_task-Baseline_.xdf
Attempting to load XDF: /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 3/sub-3_ses-1_task-Baseline_.xdf
BVP stream found and BVP0 channel extracted from /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 3/sub-3_ses-1_task-Baseline_.xdf.
UnityMarkers stream found in /home/mitchell/Documents/Projects/P8-Project/Dat

...

Task HRV calculated for 5.

Processing data for subject ID: 6 in folder: Sub 6
Processing BASELINE for subject 6 from: sub-6_ses-6_task-Baseline_.xdf
Attempting to load XDF: /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/Sub 6/sub-6_ses-6_task-Baseline_.xdf
BVP stream found and BVP0 channel extracted from /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/Sub 6/sub-6_ses-6_task-Baseline_.xdf.
UnityMarkers stream found in /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/Sub 6/sub-6_ses-6_task-Baseline_.xdf.
UnityMarkers content for baseline file (sub-6_ses-6_task-Baseline_.xdf if baseline_filepath else 'N/A'):
           time          event
0  74066.928302  BaselineStart
Calculating baseline HRV for 6 (full file)...
Baseline HRV calculated for 6.
Processing TASK for subject 6, Condition: MediumFi from: sub-6_ses-6_task-MediumFi_.xdf
Attempting to load XDF: /home/mitchell/Documents/Projects/P8-Proje

/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/neurokit2/signal/signal_interpolate.py:117: NeuroKitWarning: Duplicate x values detected. Averaging their corresponding y values.
  warn(


Task HRV calculated for 6.

Processing data for subject ID: 7 in folder: sub 7
Processing BASELINE for subject 7 from: sub-7_ses-7_task-Baseline_.xdf
Attempting to load XDF: /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 7/sub-7_ses-7_task-Baseline_.xdf
BVP stream found and BVP0 channel extracted from /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 7/sub-7_ses-7_task-Baseline_.xdf.
UnityMarkers stream found in /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 7/sub-7_ses-7_task-Baseline_.xdf.
UnityMarkers content for baseline file (sub-7_ses-7_task-Baseline_.xdf if baseline_filepath else 'N/A'):
           time          event
0  75798.067068  BaselineStart
Calculating baseline HRV for 7 (full file)...


/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/neurokit2/signal/signal_interpolate.py:117: NeuroKitWarning: Duplicate x values detected. Averaging their corresponding y values.
  warn(


Baseline HRV calculated for 7.
Processing TASK for subject 7, Condition: LowFi from: sub-7_ses-7_task-LowFi_.xdf
Attempting to load XDF: /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 7/sub-7_ses-7_task-LowFi_.xdf
BVP stream found and BVP0 channel extracted from /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/sub 7/sub-7_ses-7_task-LowFi_.xdf.
Calculating task HRV for 7 (full file 451.92s)...
Task HRV calculated for 7.

--- Combined Results DataFrame ---
  ParticipantID Condition     Phase  HRV_MeanNN     HRV_SDNN  HRV_SDANN1  \
0             1    HighFi  Baseline  715.679284   226.257083   48.481917   
1             1    HighFi      Task  727.953191  1189.775585  273.490521   
2             2    HighFi  Baseline  565.038654    57.817954   16.013175   
3             2    HighFi      Task  620.542249   684.345599  130.463341   
4             3  MediumFi  Baseline  534.085485   168.454020   27.174945   

   HRV_SDNNI1  HR

In [14]:
print(final_df)

   ParticipantID Condition     Phase  HRV_MeanNN     HRV_SDNN  HRV_SDANN1  \
0              1    HighFi  Baseline  715.679284   226.257083   48.481917   
1              1    HighFi      Task  727.953191  1189.775585  273.490521   
2              2    HighFi  Baseline  565.038654    57.817954   16.013175   
3              2    HighFi      Task  620.542249   684.345599  130.463341   
4              3  MediumFi  Baseline  534.085485   168.454020   27.174945   
5              3  MediumFi      Task  579.257756   291.470507   23.792630   
6              4    HighFi  Baseline  546.104511    71.196034   14.995208   
7              4    HighFi      Task  539.913706    63.052132    7.936986   
8              5    HighFi      Task  728.391420  1377.793085   35.635797   
9              6  MediumFi  Baseline  720.362129   343.066159   74.287398   
10             6  MediumFi      Task  708.597926  1232.896508  363.475236   
11             7     LowFi  Baseline  778.079340  1238.519766  167.767156   

# Statistical Modeling\n
\n
We now have a DataFrame (`final_df`) containing HRV metrics for each participant, condition, and phase. We can use this to perform a mixed-design analysis.\n
\n
A Linear Mixed-Effects Model (LMM) is suitable here. It allows us to model:\n
- **Fixed Effects:** The average effects of `Phase` (Baseline vs. Task) and `Condition` (LowFi, MedFi, HighFi), and their interaction (`Phase * Condition`).\n
- **Random Effects:** The variability between participants. We assume each participant has their own baseline level of the HRV metric, modeled as a random intercept (`1|ParticipantID`).\n
\n
We will model one HRV metric at a time, for example, RMSSD (Root Mean Square of Successive Differences).

In [21]:
import statsmodels.formula.api as smf

# Check if final_df exists and has data
if 'final_df' in locals() and not final_df.empty and 'HRV_RMSSD' in final_df.columns:
    # Ensure necessary columns are not all NaN
    if final_df[['HRV_RMSSD', 'Phase', 'Condition', 'ParticipantID']].isnull().all().any():
        print("Warning: One or more critical columns contain only NaN values. Cannot run model.")
    else: 
        # Remove rows with NaN in the outcome variable or predictors
        model_df = final_df.dropna(subset=['HRV_RMSSD', 'Phase', 'Condition', 'ParticipantID'])
        
        if model_df.empty:
            print("Warning: No valid data remaining after removing NaNs. Cannot run model.")
        else:
            print("\n--- Inspecting model_df before fitting LMM ---")
            print(model_df.info())

            # Print a few rows to see actual values
            print("\n--- Fitting Linear Mixed-Effects Model for HRV_RMSSD ---")
            # Ensure ParticipantID, Phase, and Condition are appropriate types
            model_df['ParticipantID'] = model_df['ParticipantID'].astype('category')
            model_df['Phase'] = model_df['Phase'].astype('category')
            model_df['Condition'] = model_df['Condition'].astype('category')
            
            # Define the model formula for fixed effects
            fixed_effects_formula = "HRV_RMSSD ~ C(Phase) * C(Condition)"
            # Define the random effects formula (random intercept for ParticipantID)
            random_effects_formula = "~1"
            
            try:
                # Fit the model using re_formula for random effects
                model = smf.mixedlm(fixed_effects_formula, 
                                  model_df, 
                                  groups=model_df["ParticipantID"], 
                                  re_formula=random_effects_formula)
                result = model.fit()
                
                # Print the summary
                print(result.summary())
            except Exception as e:
                print(f"Error fitting model: {e}")
                print("\nPlease check data structure and variability.")
                print("Model DataFrame head:")
                print(model_df.head())
else:
    print("Skipping statistical analysis: 'final_df' not created or is empty or missing 'HRV_RMSSD' column.")


--- Inspecting model_df before fitting LMM ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 94 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   ParticipantID                 13 non-null     object 
 1   Condition                     13 non-null     object 
 2   Phase                         13 non-null     object 
 3   HRV_MeanNN                    13 non-null     float64
 4   HRV_SDNN                      13 non-null     float64
 5   HRV_SDANN1                    13 non-null     float64
 6   HRV_SDNNI1                    13 non-null     float64
 7   HRV_SDANN2                    12 non-null     float64
 8   HRV_SDNNI2                    12 non-null     float64
 9   HRV_SDANN5                    0 non-null      float64
 10  HRV_SDNNI5                    0 non-null      float64
 11  HRV_RMSSD                     13 non-null     float64
 12  HRV_SDSD          

# Interpretation (Example for RMSSD)\n
\n
Look at the model summary table:\n
- **Intercept:** Estimated RMSSD for the reference group (e.g., Baseline phase in the LowFi condition, depending on how statsmodels encodes categories).\n
- **C(Phase)[T.Task]:** The estimated average *change* in RMSSD when moving from Baseline to Task phase (holding Condition constant at the reference level).\n
- **C(Condition)[T.MedFi/HighFi]:** The estimated average *difference* in RMSSD between MedFi/HighFi and the reference condition (LowFi) during the reference phase (Baseline).\n
- **C(Phase)[T.Task]:C(Condition)[T.MedFi/HighFi]:** The interaction effect. This shows how the *effect of Phase* (Task vs. Baseline) differs between the MedFi/HighFi conditions compared to the LowFi condition. A significant interaction suggests the task effect depends on the fidelity level.\n
- **P>|z|:** The p-value for each coefficient. Values < 0.05 typically indicate statistical significance.\n
- **Group Var:** The variance of the random intercepts, indicating how much baseline RMSSD varies between participants.\n
\n
*Note: You would repeat the modeling step for other HRV metrics of interest (e.g., SDNN, MeanNN, pNN50) by changing the dependent variable in the formula.*